In [1]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-code-branch" style="color:#18ab4b"></i> SageMaker Pipeline: process, then train, locally

A SageMaker SDK v3 pipeline on a local pipeline session: a <i class="fas fa-cog" style="color:#18ab4b"></i> **ProcessingStep** (`prep.py`) generates the <i class="fas fa-database" style="color:#18ab4b"></i> credit data and a <i class="fas fa-user-graduate" style="color:#18ab4b"></i> **TrainingStep** built from a `ModelTrainer` fits the CatBoost model on it, wired together as on AWS, with inter-step artifacts passing through oblako's **S3Proxy**. Mirrors `batch_scoring/.../sagemaker_pipeline.py`.

Prereqs: `oblako up s3`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

In [2]:
import logging

# the SageMaker SDK logs every step at INFO through a root rich handler, unless the
# root logger is configured first: keep its output (and botocore's) to warnings
logging.basicConfig(level=logging.WARNING)
logging.getLogger("sagemaker.config").setLevel(logging.WARNING)
# a local endpoint is polled on /ping while it starts; the retries aren't news
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

## <i class="fas fa-database" style="color:#18ab4b"></i> Wire S3Proxy as the artifact store + <i class="fas fa-file-code" style="color:#6c63ff"></i> build the container

In [3]:
import os

from oblako.engines.sagemaker import use_local_stubs
from oblako.services import S3ProxyService, SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
# SageMaker passes step artifacts via S3: point the SDK's boto3 clients at S3Proxy,
# which doesn't implement botocore's default flexible checksums
os.environ["AWS_ENDPOINT_URL_S3"] = "http://localhost:9000"
os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"
use_local_stubs()  # no AWS account: skip the SDK's IAM role check

IMAGE = "oblako-credit-scoring:latest"
ROLE = "arn:aws:iam::000000000000:role/dummy"
BUCKET = "credit-scoring-pipeline"

SageMakerService().build_image(path="container", tag=IMAGE)
s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket=BUCKET)
except Exception:
    pass
# start from an empty io/ prefix so the listing below shows this run only
for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix="io/").get("Contents", []):
    s3.delete_object(Bucket=BUCKET, Key=obj["Key"])

Step 1/7 : FROM python:3.11-slim
 ---> 1042b61448fe
Step 2/7 : RUN apt-get update && apt-get install -y --no-install-recommends libgomp1     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> 2e7552efc476
Step 3/7 : RUN pip install --no-cache-dir catboost numpy pandas joblib
 ---> Using cache
 ---> ded82ae17d1d
Step 4/7 : COPY src/ /opt/program/
 ---> Using cache
 ---> 5ceb17bd1b9f
Step 5/7 : WORKDIR /opt/program
 ---> Using cache
 ---> aec1e1cacaa1
Step 6/7 : ENV PYTHONUNBUFFERED=1
 ---> Using cache
 ---> bfec09cf3631
Step 7/7 : ENTRYPOINT ["python", "main.py"]
 ---> Using cache
 ---> 8982120980d7
Successfully built 8982120980d7
Successfully tagged oblako-credit-scoring:latest


## <i class="fas fa-code-branch" style="color:#18ab4b"></i> Define and run the pipeline

SDK v3 splits the local pipeline session in two: `sagemaker.mlops` has the pipeline methods, while `PipelineSession` makes `run()` / `train()` return step arguments instead of executing. The class below combines both. Locally the pipeline is registered and started through the session; `pipeline.upsert()` / `start()` are the calls to use on AWS.

In [4]:
from sagemaker.core.processing import (
    PipelineSession,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.core.shapes import ProcessingS3Output
from sagemaker.mlops.local.local_pipeline_session import (
    LocalPipelineSession as _MlopsLocalPipelineSession,
)
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute, InputData, OutputDataConfig


class LocalPipelineSession(_MlopsLocalPipelineSession, PipelineSession):
    """The mlops pipeline methods plus PipelineSession's deferred steps."""


session = LocalPipelineSession(default_bucket=BUCKET)

processor = ScriptProcessor(
    image_uri="python:3.11-slim",
    command=["python3"],
    instance_type="local",
    instance_count=1,
    base_job_name="generate-data",
    role=ROLE,
    sagemaker_session=session,
)
prep = ProcessingStep(
    name="GenerateData",
    step_args=processor.run(
        code="prep.py",
        outputs=[
            ProcessingOutput(
                output_name="train",
                s3_output=ProcessingS3Output(
                    s3_uri=f"s3://{BUCKET}/io/prep/train",
                    local_path="/opt/ml/processing/output",
                    s3_upload_mode="EndOfJob",
                ),
            )
        ],
    ),
)

trainer = ModelTrainer(
    training_image=IMAGE,
    role=ROLE,
    base_job_name="train-credit",
    compute=Compute(instance_type="local", instance_count=1),
    output_data_config=OutputDataConfig(s3_output_path=f"s3://{BUCKET}/io/train"),
    hyperparameters={
        "iterations": "150",
        "depth": "6",
        "learning-rate": "0.1",
        "factor": "20",
        "offset": "600",
    },
    sagemaker_session=session,
)
train_step = TrainingStep(
    name="TrainCreditModel",
    step_args=trainer.train(
        input_data_config=[
            InputData(
                channel_name="train",
                data_source=prep.properties.ProcessingOutputConfig.Outputs[
                    "train"
                ].S3Output.S3Uri,
            )
        ],
        wait=False,
    ),
)

pipeline = Pipeline(
    name="oblako-credit-scoring", steps=[prep, train_step], sagemaker_session=session
)
session.create_pipeline(pipeline, "credit scoring: generate data, then train")
execution = session.start_pipeline_execution(PipelineName=pipeline.name)
print("pipeline:", execution.describe()["PipelineExecutionStatus"])
for step in execution.list_steps()["PipelineExecutionSteps"]:
    print(" ", step["StepName"], step["StepStatus"])
print(
    "artifacts:",
    [
        o["Key"]
        for o in s3.list_objects_v2(Bucket=BUCKET, Prefix="io/").get("Contents", [])
    ],
)

time="2026-10-04T19:42:07+02:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmp1g8aosm3\".\nSet `external: true` to use an existing network"
 Container aexnbf0s1s-algo-1-vy8hh Creating 
 Container aexnbf0s1s-algo-1-vy8hh Created 
Attaching to aexnbf0s1s-algo-1-vy8hh
 Container aexnbf0s1s-algo-1-vy8hh Starting 
 Container aexnbf0s1s-algo-1-vy8hh Started 


aexnbf0s1s-algo-1-vy8hh  | wrote /opt/ml/processing/output/train.csv
aexnbf0s1s-algo-1-vy8hh exited with code 0
 Compose Stopping Aborting on container exit...
 Container aexnbf0s1s-algo-1-vy8hh Stopping 
 Container aexnbf0s1s-algo-1-vy8hh Stopped 


pipeline: Failed
  GenerateData Succeeded
  TrainCreditModel Failed
artifacts: ['io/prep/train/train.csv']
